# Reproduce the TSP paper — canonical run / artifact mode
**Upload-and-run.** Two modes:
- **`RETRAIN=True` (canonical run, GPU):** trains every trainable detector at the
  TSP budget (≥2000 epochs, `mse_optimal` σ), scores the full amplitude grid
  (dense in (0,0.3], sparse in (0.3,1]) + the strong replacement cell, on all
  three scenes. Checkpoint-resume per (detector, scene, signature, seed): a
  disconnected session loses at most one cell. Ends with an auto-downloaded zip.
- **`RETRAIN=False` (artifact mode):** downloads the published results zip and
  regenerates every table/figure from raw scores in minutes, no training.

Locally, feed the downloaded zip to `python -m tsp_repro.ingest_zip --zip ... --tsp ...`
to write the paper's `tables/*.tex` and `figures/*.pdf`.


In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, torch
sys.path.insert(0, '.')
import tsp_repro  # installs the vendor path shims
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
# ---- configuration ----
RETRAIN = True          # False = artifact mode (download published results)
MOUNT_DRIVE = False     # True: keep ckpts/results on Drive across disconnects

CKPT_DIR, OUT_DIR = 'ckpt_tsp', 'results_tsp'
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/tsp_repro/ckpt_tsp'
    OUT_DIR  = '/content/drive/MyDrive/tsp_repro/results_tsp'
import os; os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

from tsp_repro import protocol as PR
from tsp_repro.registry import OUR_DETECTORS, CLASSICAL, DEEP, REGISTRY
print('theta grid:', PR.THETA_GRID)
print('epochs   :', PR.EPOCHS)


## Artifact mode (skip if retraining)


In [ ]:
if not RETRAIN:
    from tsp_repro.artifacts import fetch_release
    fetch_release()  # -> tsp_artifacts/; tables/figures cells below will find it


## Canonical run — our detectors + classical baselines
Order of scenes: Pavia (4 signatures) is the long one; San Diego I–II are quick.
Rough T4 budget: DARTS 2000 ep ≈ 25–35 min/seed on Pavia; DART/LRao minutes.


In [ ]:
OURS = OUR_DETECTORS + CLASSICAL   # DART(+CFAR), DARTS(+CFAR), AMF, AMF-local, GMM-Levin, LRao
if RETRAIN:
    from tsp_repro.runner import run_scene
    res_sd1 = run_scene('sandiego',  OURS, out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, device=DEVICE)
    res_sd2 = run_scene('sandiego2', OURS, out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, device=DEVICE)
    res_pav = run_scene('pavia4',    OURS, out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, device=DEVICE)


## Canonical run — deep baselines (paper-faithful ports)
THANTD + HTD-Net run here; OS-VAE and TSTTD are artifact-only until their
runnable ports are wired (Phase B; raw scores from the validated MLSP runs
are used in the tables meanwhile).


In [ ]:
if RETRAIN:
    from tsp_repro.runner import run_scene
    run_scene('pavia4', ['THANTD', 'HTDNet'], sig_label='bitumen',
              out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, device=DEVICE)


## Sanity gate vs the MLSP archive
The 2000-epoch canonical numbers at θ=0.15 must be consistent with the
1000-epoch MLSP archive (flag any detector moving > 0.02 AUC).


In [ ]:
MLSP_REF = {  # Pavia scene 4, bitumen, theta=0.15 (MLSP archive, 5 seeds)
    'DART': 0.789, 'DART-CFAR': 0.868, 'DARTS': 0.930, 'DARTS-CFAR': 0.940,
    'AMF': 0.773, 'AMF-local': 0.839, 'GMM-Levin': 0.810, 'LRao': 0.744}
import numpy as np, json, os
p = os.path.join(OUT_DIR, 'pavia4__bitumen__results.json')
if os.path.exists(p):
    res = json.load(open(p))
    for det, ref in MLSP_REF.items():
        v = res.get(det, {}).get('additive|0.15')
        if v is None: continue
        m = float(np.mean(v)); flag = '  <-- CHECK' if abs(m - ref) > 0.02 else ''
        print(f'{det:12s} tsp={m:.3f}  mlsp={ref:.3f}{flag}')


## Package + download everything


In [ ]:
from tsp_repro.runner import zip_results
zip_results(dirs=(OUT_DIR, CKPT_DIR), zip_name='tsp_results.zip')


## Preview tables/figures in-notebook (works in both modes)


In [ ]:
from tsp_repro import tables as T, figures as F
from tsp_repro.artifacts import scores_dirs
dirs = [OUT_DIR] if RETRAIN else scores_dirs()
dirs = [d for d in dirs if os.path.isdir(d)]
cells = T.collect(dirs)
dets = sorted({k[2] for k in cells})
print(f'{len(cells)} cells from {dirs}; detectors: {dets}')
os.makedirs('preview', exist_ok=True)
T.write_generality_tables('preview', dirs, dets)
F.amp_sweep('preview/amp_sweep.pdf', dirs, dets)
F.scenes_falsecolor('preview/scenes_falsecolor.pdf')
